In [3]:
import torch
import torchvision.models as models
import torch.nn as nn

# 1. 加载预训练ResNet18（ImageNet预训练权重）
model = models.resnet18(pretrained=True)

# ========== 对应问题1：分层参数设置 ==========
# 拆分：底层特征提取层 + 顶层全新分类输出层
feature_layers = list(model.children())[:-1]  # 除最后全连接层之外的所有卷积BN主干
fc_layer = model.fc                         # 顶层输出全连接层

# 方式A：冻结主干特征层（学习率=0，完全不更新）
for param in model.parameters():
    param.requires_grad = False
# 仅解冻顶层输出层
for param in fc_layer.parameters():
    param.requires_grad = True

# 替换为新数据集分类头（假设新数据集10类）
in_dim = fc_layer.in_features
model.fc = nn.Linear(in_dim, 10)

# ========== 分层设置不同学习率 ==========
# 特征提取层：极小学习率 1e-5；顶层输出层：大学习率 1e-3
optimizer = torch.optim.SGD([
    {"params": feature_layers[-1].parameters(), "lr": 1e-5},   # 底层小lr
    {"params": model.fc.parameters(), "lr": 1e-3}              # 顶层大lr
], momentum=0.9)

# 打印验证每层学习率
print("特征提取层学习率:", optimizer.param_groups[0]["lr"])
print("顶层输出层学习率:", optimizer.param_groups[1]["lr"])

# ========== 对应问题2：小相似数据集防过拟合代码配套方案 ==========
# 1. 全程冻结主干，仅训练分类头（最稳妥）
# 2. 添加正则化：Dropout、weight_decay权重衰减
model.fc = nn.Sequential(
    nn.Dropout(0.3),          # Dropout防过拟合
    nn.Linear(in_dim, 10)
)
# 优化器开启L2正则
optimizer_reg = torch.optim.SGD([
    {"params": feature_layers[-1].parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-3, "weight_decay": 1e-4}
], momentum=0.9)

print("\n开启L2正则+Dropout的优化器构建完成，用于小样本微调防过拟合")

c:\Users\86172\Desktop\深度学习课程设计\myenv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\86172\Desktop\深度学习课程设计\myenv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\86172/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


特征提取层学习率: 1e-05
顶层输出层学习率: 0.001

开启L2正则+Dropout的优化器构建完成，用于小样本微调防过拟合


In [4]:
from torchvision import transforms
from PIL import Image

# 训练集增广流水线，严格匹配题目4条要求
train_transform = transforms.Compose([
    # 1. 随机裁剪面积0.08~1.0，resize到224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2. 50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3. 亮度、对比度、饱和度抖动范围0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    # 4. 转为torch张量
    transforms.ToTensor()
])

# 测试代码
if __name__ == "__main__":
    test_img = Image.new("RGB", (256, 256), (100, 150, 200))
    tensor_img = train_transform(test_img)
    print("增广后张量形状 [C, H, W] =", tensor_img.shape)

增广后张量形状 [C, H, W] = torch.Size([3, 224, 224])
